# Python object-oriented programming

---

1. [Inheritance](),
    - [meaning of the term](),
    - [simple inheritance](),
    - [super() function](),
    - [multiple inheritance](),
    - [chained inheritance](),
    - [method resolution order](),
2. [Abstraction](),
    - [meaning of the term](),
    - [in practice](),
    - [ABC library](),
3. [Data classes](),
    - [default values](),
    - [validation via magic method]().

<br>

---

<br>

## Inheritance


### Meaning of the term

---

In general, **inheritance** means some reference or legacy.

In OOP **Inheritance** is the mechanism that lets you transfer *attributes* and *methods* from **one class** (the *parent*) to **other classes** (the *children*).

<br>

Essentially, every user-defined class is a subclass of the `object` class:

In [1]:
class MyClass:
    pass

In [2]:
class MyClass(object):
    pass

In [3]:
print(issubclass(MyClass, object))  # object --> MyClass --> bool

True


<br>

In practice you hardly notice this, because the **interpreter** lets you omit the parent by default:

In [4]:
class Employee:
    pass

In [5]:
print(issubclass(Employee, object))

True


<br>

**Standard inheritance** is not always what you need.

For example, when you need to create **custom classes** with custom parents:

In [6]:
class Employee:
    """Object for creating a regular employee."""
    
    def __init__(self, name: str, age: int, salary: int):
        self.name = name
        self.age = age
        self.salary = salary
        
    def create_email(self, domain: str) -> str:
        self.email = f"{self.name.lower()}{domain}"

In [7]:
matthew = Employee("Matthew", 40, 80_000)

In [8]:
matthew.create_email("@superdomain.com")

In [9]:
print(matthew.__dict__)

{'name': 'Matthew', 'age': 40, 'salary': 80000, 'email': 'matthew@superdomain.com'}


<br>

You have a class that controls the process of creating new instances **for employees**.

But what if you need another class, this time for *testers*:

In [10]:
class Tester:
    """Object for creating an employee who is a tester."""
    
    def __init__(self, name: str, age: int, salary: int):
        self.name = name
        self.age = age
        self.salary = salary
        self.__vcs_access = False

    def create_email(self):
        self.email = f"{self.name.lower()}{domain}"
    
    @property
    def vcs_access(self):
        return self.__vcs_access
    
    @vcs_access.setter
    def vcs_access(self, value: bool):
        if value in {True, False}:
            self.__vcs_access = value
        else:
            raise Exception("Cannot assign such a value")

In [11]:
peter = Tester("Peter", 45, 100_000)

In [12]:
peter.vcs_access = True

In [13]:
peter.vcs_access

True

In [14]:
print(peter.__dict__)

{'name': 'Peter', 'age': 45, 'salary': 100000, '_Tester__vcs_access': True}


### Simple inheritance

---

<br>

At first glance you can see that having **two similar classes** (`Employee` and `Tester`) side by side is not efficient.

It goes against the software development principle (*DRY - don't repeat yourself*).

<br>

For this specific role, a lot of code had to be duplicated.

Most of the `Tester` class repeats the definition of the `Employee` class.

For this reason you can use another, third concept that OOP is built on: *Inheritance*:

In [15]:
class Tester(Employee):
    """Object for creating an employee who is a tester."""
    
    @property
    def vcs_access(self):
        return self.__vcs_access
    
    @vcs_access.setter
    def vcs_access(self, value: bool):
        if value in {True, False}:
            self.__vcs_access = value
        else:
            raise Exception("Cannot assign such a value")

In [16]:
luke = Tester("Luke", 35, 75_000)

In [17]:
luke.vcs_access = True

In [18]:
print(luke.vcs_access)

True


In [19]:
luke.__dict__

{'name': 'Luke', 'age': 35, 'salary': 75000, '_Tester__vcs_access': True}

In [20]:
email = luke.create_email("@superdomain.com")

In [21]:
print(luke.email)

luke@superdomain.com


<br>

This way you write less code and reuse the existing parent class.

You can confirm with a built-in function that the `Tester` class is a subclass of `Employee`:

In [22]:
print(issubclass(Tester, Employee))

True


<br>

You can also notice that there is no need to repeat the *instance attribute* definitions.

In [23]:
class Employee:
    
    def __init__(self, name: str, age: int, salary: int) -> None:
        self.name = name
        self.age = age
        self.salary = salary

In [24]:
class Tester(Employee):
    pass

In [25]:
tester_matthew = Tester("Matthew", 51, 50_000)

In [26]:
print(tester_matthew.__dict__)

{'name': 'Matthew', 'age': 51, 'salary': 50000}


<br>

If you need to add a **specific instance attribute** for the child class `Tester`, you must work with it carefully:

In [27]:
class Employee:
    
    def __init__(self, name: str, age: int, salary: int) -> None:
        self.name = name
        self.age = age
        self.salary = salary

In [28]:
class Tester(Employee):
    
    def __init__(self, vcs_access: bool):
        self.vcs_access = vcs_access

In [29]:
tester_peter = Tester("Peter", 50, 55_000)

TypeError: Tester.__init__() takes 2 positional arguments but 4 were given

In [30]:
tester_peter = Tester(False)

A child written like this—with some attributes from the parent and some from the child—cannot be used this way.

The new constructor `__init__` completely overrides the parent:

In [31]:
class Tester(Employee):
    def __init__(self, name: str, age: int, salary: str, vcs_access: bool):
        self.name = name
        self.age = age
        self.salary = salary
        self.vcs_access = vcs_access

In [32]:
tester_peter = Tester("Peter", 50, 55_000, True)

In [33]:
print(tester_peter.__dict__)

{'name': 'Peter', 'age': 50, 'salary': 55000, 'vcs_access': True}


There is, however, a friendlier and more readable solution.

### The super() function

----

<br>

What if you want to **modify an existing behaviour from the parent**:

In [34]:
class Parent:
    def function_x(self, x):
        print("Parent", self, x)

In [35]:
class Child(Parent):
    def function_x(self, x):
        print("Child", self, x)
        print("Parent", self, x)  # I want to avoid duplicating code
        print("End of Child class")

In [36]:
p = Child()
p.function_x(13)

Child <__main__.Child object at 0x000002637CAC2750> 13
Parent <__main__.Child object at 0x000002637CAC2750> 13
End of Child class


<br>

In the `Child` class you can notice duplicated statements.

At the same time it also contains its own.

You can simplify this using the `super()` function:

In [37]:
class Parent:
    def function_x(self, x):
        print("Parent", self, x)

In [38]:
class Child(Parent):
    def function_x(self, x):
        print("Child", self, x)
        super().function_x(x)  # __init__()
        print("End of Child class")

In [39]:
p = Child()

In [40]:
p.function_x(13)

Child <__main__.Child object at 0x000002637C5658D0> 13
Parent <__main__.Child object at 0x000002637C5658D0> 13
End of Child class


<br>

The `super()` function automatically refers to the next parent in the chain.

This makes the code less verbose and less error-prone.

With `super()` you don't need to pass the `self` or `cls` parameter (the interpreter adds it automatically).

<img src="https://i.imgur.com/ZxAEWmu.png" width="1000" style="margin-left:auto; margin-right:auto">

Back to the previous example with the `Employee` and `Tester` classes:

In [41]:
class Tester(Employee):
    def __init__(self, name: str, age: int, salary: str, vcs_access: bool):
        super().__init__(name, age, salary)
        self.vcs_access = vcs_access

In [42]:
tester_peter = Tester("Peter", 50, 55_000, True)

In [43]:
print(tester_peter.__dict__)

{'name': 'Peter', 'age': 50, 'salary': 55000, 'vcs_access': True}


### Multiple inheritance

---

<img src="https://i.imgur.com/ckZ0gpi.png" width="1000" style="margin-left:auto; margin-right:auto">

Inheritance does not have to be **single-level** only.

One parent can have **multiple children**:

In [44]:
class Employee:
    """Object for creating a regular employee."""
    
    def __init__(self, name: str, age: int, salary: int):
        self.name = name
        self.age = age
        self.salary = salary
        
    def create_email(self, domain: str) -> str:
        self.email = f"{self.name.lower()}{domain}"

In [45]:
class Tester(Employee):
    """Object for creating an employee who is a tester."""
    
    @property
    def vcs_access(self):
        return self.__vcs_access
    
    @vcs_access.setter
    def vcs_access(self, value: bool):
        if value in {True, False}:
            self.__vcs_access = value
        else:
            raise Exception("Cannot assign such a value")

<br>

Additionally, add another class named `BigDataEngineer` that, like `Tester`, inherits from `Employee`:

In [46]:
class BigDataEngineer(Employee):
    """Object for creating an employee who is a big data engineer."""
    
    @property
    def db_access(self):
        return self.__db_access
    
    @db_access.setter
    def db_access(self, value: bool):
        if value in {True, False}:
            self.__db_access = value
        else:
            raise Exception("Cannot assign such a value")

In [47]:
filip = BigDataEngineer("Filip", 40, 120_000)

In [48]:
filip.db_access = True

In [49]:
filip.create_email("@gmail.com")

In [50]:
print(filip.__dict__)

{'name': 'Filip', 'age': 40, 'salary': 120000, '_BigDataEngineer__db_access': True, 'email': 'filip@gmail.com'}


<br>

At this point the child `BigDataEngineer` only works with its own methods, or those of its parent.

What if you want to use methods from a **sibling class**:

In [51]:
filip.vcs_access = True

In [52]:
filip.vcs_access

True

<br>

How is that possible? Siblings don't share each other's attributes:

In [53]:
print(filip.__dict__)

{'name': 'Filip', 'age': 40, 'salary': 120000, '_BigDataEngineer__db_access': True, 'email': 'filip@gmail.com', 'vcs_access': True}


In [54]:
print(
    issubclass(BigDataEngineer, Employee),
    issubclass(Tester, Employee),
    issubclass(BigDataEngineer, Tester),
    issubclass(Tester, BigDataEngineer),
    sep="\n"
)

True
True
False
False


<br>

The assignment above essentially creates a **new instance attribute**:

In [55]:
filip.cloud_access = True

In [56]:
print(filip.__dict__)

{'name': 'Filip', 'age': 40, 'salary': 120000, '_BigDataEngineer__db_access': True, 'email': 'filip@gmail.com', 'vcs_access': True, 'cloud_access': True}


<br>

That of course makes no sense and can also break the functionality of the classes.

### Chained inheritance

---

<img src="https://i.imgur.com/S9iKJJ5.png" width="1000" style="margin-left:auto; margin-right:auto">

Unlike **multiple inheritance**, chained inheritance is often undesirable:

In [57]:
class Grandparent:
    def function_x(self, x):
        print("Grandparent", self, x)

In [58]:
class Parent(Grandparent):
    def function_x(self, x):
        print("Parent", self, x)
        super().function_x(x)

In [59]:
class Child(Parent):
    def function_x(self, x):
        print("Child", self, x)
        super().function_x(x)

In [60]:
p = Child()

In [61]:
p.function_x(14)

Child <__main__.Child object at 0x000002637CADFBD0> 14
Parent <__main__.Child object at 0x000002637CADFBD0> 14
Grandparent <__main__.Child object at 0x000002637CADFBD0> 14


<br>

*Linear chained inheritance* is possible, but it quickly becomes **hard to follow and maintain**.

If you need to use this pattern, think carefully about whether the code is still **clear and understandable**.

<br>

Over time you may want to add a new type of employee; again you can apply inheritance:

In [62]:
class Employee:
    """Object for creating a regular employee."""
    
    def __init__(self, name: str, age: int, salary: int):
        self.name = name
        self.age = age
        self.salary = salary
        
    def create_email(self, domain: str) -> str:
        self.email = f"{self.name.lower()}{domain}"

In [63]:
class Tester(Employee):
    """Object for creating an employee who is a tester."""
    
    @property
    def vcs_access(self):
        return self.__vcs_access
    
    @vcs_access.setter
    def vcs_access(self, value: bool):
        if value in {True, False}:
            self.__vcs_access = value
        else:
            raise Exception("Cannot assign such a value")

In [64]:
class BigDataEngineer(Tester):
    """Object for creating an employee who is a big data engineer."""
    
    @property
    def db_access(self):
        return self.__db_access
    
    @db_access.setter
    def db_access(self, value: bool):
        if value in {True, False}:
            self.__db_access = value
        else:
            raise Exception("Cannot assign such a value")

In [65]:
tom = BigDataEngineer("Tom", 31, 70_000)

In [66]:
tom.db_access = True

In [67]:
tom.vcs_access = True

In [68]:
print(
    tom.name,
    tom.db_access,
    tom.vcs_access,
    sep="\n"
)

Tom
True
True


In [69]:
print(tom.__dict__)

{'name': 'Tom', 'age': 31, 'salary': 70000, '_BigDataEngineer__db_access': True, '_Tester__vcs_access': True}


In [70]:
tom.create_email("@gmail.com")

In [71]:
print(tom.__dict__)

{'name': 'Tom', 'age': 31, 'salary': 70000, '_BigDataEngineer__db_access': True, '_Tester__vcs_access': True, 'email': 'tom@gmail.com'}


<br>

It really is no longer easy to trace where changes come from.

And in this example we're dealing with trivial classes.

<br>

What if inheritance gets even more complicated?

<img src="https://i.imgur.com/HgQpT9I.png" width="1000" style="margin-left:auto; margin-right:auto">

### MRO (Method resolution order)

---

If the inheritance hierarchy has many levels, it would not be easy to follow the parent chain.

For this reason there is a method that shows the order in which classes are searched:

In [72]:
class Employee:
    def f(self):
        print("Employee", self)

class FrontendDev(Employee):
    def f(self):
        print("FrontendDev", self)
        super().f()

class BackendDev(Employee):
    def f(self):
        print("BackendDev", self)
        super().f()
        
class FullstackDev(FrontendDev, BackendDev):
    def f(self):
        print("FullstackDev", self)
        super().f()

In [73]:
fdev = FullstackDev()

In [74]:
fdev.f()

FullstackDev <__main__.FullstackDev object at 0x000002637D0BF6D0>
FrontendDev <__main__.FullstackDev object at 0x000002637D0BF6D0>
BackendDev <__main__.FullstackDev object at 0x000002637D0BF6D0>
Employee <__main__.FullstackDev object at 0x000002637D0BF6D0>


<br>

**MRO** is also available as a *magic attribute*, so you can work with it directly:

In [75]:
FullstackDev.__mro__

(__main__.FullstackDev,
 __main__.FrontendDev,
 __main__.BackendDev,
 __main__.Employee,
 object)

<br>

Why is MRO so important?

Look at a small change in the code:

In [76]:
class Employee:
    def f(self):
        print("Employee", self)

class FrontendDev(Employee):
    def f(self):
        print("FrontendDev", self)
        super().f()

class BackendDev(Employee):
    def f(self):
        print("BackendDev", self)
        # super().f()
                
class FullstackDev(FrontendDev, BackendDev):
    def f(self):
        print("FullstackDev", self)
        super().f()

In [77]:
fdev = FullstackDev()

In [78]:
fdev.f()

FullstackDev <__main__.FullstackDev object at 0x000002637D0BD990>
FrontendDev <__main__.FullstackDev object at 0x000002637D0BD990>
BackendDev <__main__.FullstackDev object at 0x000002637D0BD990>


<br>

It follows that `super()` does not work directly with the parent or sibling.

It works with the next class in the order defined by MRO:

In [79]:
class Employee:
    atri = "E"

class FrontendDev(Employee):
    # pass
    atri = "Fr"

class BackendDev(Employee):
    atri = "B"
        
class FullstackDev(FrontendDev, BackendDev):
    # pass
    atri = "Fu"

In [80]:
f = FullstackDev()

In [81]:
f.atri

'Fu'

In [82]:
BackendDev.__mro__

(__main__.BackendDev, __main__.Employee, object)

<br>

MRO does not apply **only to methods**.

The same order is used when looking up attributes of classes.

### Summary

---

<br>

*Inheritance*, like other concepts, must not be **overused**.

<br>

With *chained inheritance*, the flow becomes **unclear** and the program becomes too complex.

<br>

The interpreter first looks for **methods or attributes** in the current class (the child), and only then follows the MRO.

<br>

### 🧠 EXERCISE 🧠 Create a class `Car` and add the following:

- Define a class `Car` with an `__init__` method that takes only parameters `brand`, `color`,
- create *getter* and *setter* methods for the attribute `max_speed` (maximum 200 km/h) and `weight` (maximum 2 tonnes),
- create a class `PassengerCar` that inherits from `Car` the attributes `brand`, `color` and additionally needs an instance attribute `seats`,
- for `PassengerCar` create a *getter* and *setter* that limits `max_speed` to 300 km/h and the value is stored as an integer,
- create a class `Truck` that inherits from `Car` the attributes `brand`, `color` and additionally needs an instance attribute `capacity`,
- for `Truck` create a getter and setter that limits `max_speed` to 100 km/h and the value is stored as an integer.

In [83]:
class Car:
    def __init__(self, brand, color):
        self.brand = brand
        self.color = color
        
    @property
    def max_speed(self):
        return self._max_speed
        
    @max_speed.setter
    def max_speed(self, value):
        if value <= 200:
            self._max_speed = value
        else:
            raise ValueError("Speed must not exceed 200 km/h!")
            
    @property
    def weight(self):
        return self._weight
        
    @weight.setter
    def weight(self, value):
        if value <= 2:
            self._weight = value
        else:
            raise ValueError("Weight must not exceed 2 tonnes!") 

            
class PassengerCar(Car):
    def __init__(self, brand, color, seats):
        super().__init__(brand, color)
        self.seats = seats
    
    @property
    def max_speed(self):
        return self._max_speed
        
    @max_speed.setter
    def max_speed(self, value):
        if isinstance(value, int) and value <= 300:
            self._max_speed = value
        else:
            raise ValueError(
                "Speed must not exceed 300 km/h"
                " or value is not an integer!"
            ) 
            
class Truck(Car):
    def __init__(self, brand, color, capacity):
        super().__init__(brand, color)
        self.capacity = capacity
    
    @property
    def max_speed(self):
        return self._max_speed
        
    @max_speed.setter
    def max_speed(self, value):
        if isinstance(value, int) and value <= 100:
            self._max_speed = value
        else:
            raise ValueError(
                "Speed must not exceed 100 km/h"
                " or value is not an integer!"
            )

In [84]:
car_1 = Car("BMW", "blue")
car_2 = PassengerCar("Audi", "white", 5)

In [85]:
car_1.max_speed = 200

In [86]:
print(car_1.__dict__)

{'brand': 'BMW', 'color': 'blue', '_max_speed': 200}


In [87]:
car_2.max_speed = 300

In [88]:
print(car_2.__dict__)

{'brand': 'Audi', 'color': 'white', 'seats': 5, '_max_speed': 300}


In [89]:
transit = Truck("Ford", "grey", 1.5)

In [90]:
transit.max_speed = 99

In [92]:
transit.max_speed = 150

ValueError: Speed must not exceed 100 km/h or value is not an integer!

In [91]:
print(transit.__dict__)

{'brand': 'Ford', 'color': 'grey', 'capacity': 1.5, '_max_speed': 99}


<details>
    <summary>▶️ Solution</summary>
    
```python
class Car:
def __init__(self, brand: str, color: str):
    self.brand = brand
    self.color = color
    self.__weight = 0
    self.__max_speed = 0

@property
def weight(self):
    return self.__weight

@weight.setter
def weight(self, value: int) -> None:
    if isinstance(value, int) and 0 < value < 2:
        self.__weight = value
    else:
        raise Exception("Wrong data type or value (<2 tonnes)")

@property
def max_speed(self):
    return self.__max_speed

@max_speed.setter
def max_speed(self, value: float) -> None:
    if isinstance(value, int) and 0 < value < 200:
        self.__max_speed = value
    else:
        raise Exception("Wrong data type or value (<200 km/h)")


class PassengerCar(Car):
    def __init__(self, brand: str, color: str, seats: int):
        super().__init__(brand, color)
        self.seats = seats

    @property
    def max_speed(self):
        return self.__max_speed

    @max_speed.setter
    def max_speed(self, value: float) -> None:
        if isinstance(value, int) and 0 < value < 300:
            self.__max_speed = value
        else:
            raise Exception("Wrong data type or value (<300 km/h)")


class Truck(Car):
    def __init__(self, brand: str, color: str, capacity: float):
        super().__init__(brand, color)
        self.capacity = capacity

    @property
    def max_speed(self):
        return self.__max_speed

    @max_speed.setter
    def max_speed(self, value: int) -> None:
        if isinstance(value, int) and 0 < value < 100:
            self.__max_speed = value
        else:
            raise Exception("Wrong data type or value (<100 km/h)")
```
</details>

<br>

## Abstraction

---

<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Ftse1.mm.bing.net%2Fth%3Fid%3DOIP.CFwkjtcCFee-YqXbmNDmxAHaHa%26pid%3DApi&f=1&ipt=b4737f334dd002ab30b8da0699a297e413334fd0ea059deb6cf4a6a0087b3f6b&ipo=images" width="250" style="margin-left:auto; margin-right:auto">

### Meaning of the term

---

Abstraction is about **reducing the amount of detail**.

In OOP it means that the user does not need to know all implementations, only the **names of objects**.

<br>

Just as in practice you don't know everything that happens when you press the camera button on a smartphone.

So too in Python you don't need to know all the details:

In [ ]:
"matthew".title()

In [ ]:
"OOP".lower()

### In practice

---

<br>

Similarly you can see *abstraction* in OOP.

As the author of your library you try to make life easier for other users of the library.

So they only need a clear attribute like `account_type`, which looks up and calls the specific functions/methods.

<!-- 
```python
class GooglePaymentProcessor:
    
    def __init__(self, order: int):
        self.order = order

    def pay(self):
        print(
            f"Nr.order: {self.order}",
            "Processing GooglePay..",
            "Verifying security code..",
            "Changing order status..",
            "-" * 25,
            sep="\n"
        )
```

```python
order_1 = GooglePaymentProcessor("1234567890")
order_1.pay()
``` -->

In [ ]:
class User:
    def __init__(self, name: str, email: str):
        self.email = email
        self.name = name

    @property
    def account_type(self):
        return self.__account_type
    
    @account_type.setter
    def account_type(self, type_: str):
        """docstring"""
        if isinstance(type_, str) \
            and type_.title() in ("Gold", "Platinum", "Diamond"):
            self.__account_type = type_
            print("Preparing new account...")
        else:
            raise Exception()

In [ ]:
user_1 = User("Matthew", "matthew@gmail.com")

In [ ]:
user_1.account_type = "gold"

<br>

As a user of such a library, you don't deal with the implementation of these helper functions.

You don't need to test or document them, because that is the *library author's* concern.

Ideally you just **apply the library or object** for your purposes.

### The abc library

---

Another application is not only hiding the logic of a specific object, but also **entire classes**:

In [ ]:
class User:
    def greet(self):
        print("Welcome to your account")
    
class GoldAccount(User):
    pass
        
        
class PlatinumAccount(User):
    pass

In [ ]:
user_1 = User()
user_2 = GoldAccount()
user_3 = PlatinumAccount()

In [ ]:
user_1.greet()
user_2.greet()
user_3.greet()

With *inheritance* you can use the parent's `greet` method.

In some situations that makes sense, but each account type should get an appropriate greeting.

So you need to override the original `greet` method so it fits **each child class**:

In [ ]:
class User:
    def greet(self):
        pass
    
class GoldAccount(User):
    def greet(self):
        print("Welcome to your Gold account")
        
        
class PlatinumAccount(User):
    def greet(self):
        print("Welcome to your Platinum account")

In [ ]:
user_1 = User()
user_2 = GoldAccount()
user_3 = PlatinumAccount()

In [ ]:
user_1.greet()
user_2.greet()
user_3.greet()

In such a situation you might wonder why keep the `greet` method in the parent `User` at all?

In [ ]:
class User:
    pass  # remove the method
    
class GoldAccount(User):
    def greet(self):
        print("Welcome to your Gold account")
        
        
class PlatinumAccount(User):
    def greet(self):
        print("Welcome to your Platinum account")

In [ ]:
user_1 = User()
user_2 = GoldAccount()
user_3 = PlatinumAccount()

In [ ]:
# user_1.greet()
user_2.greet()
user_3.greet()

Now the child classes `GoldAccount` and `PlatinumAccount` work the same way. Right?

In [ ]:
class User:
    pass
    
class GoldAccount(User):
    def greet(self):
        print("Welcome to your Gold account")
        
        
class PlatinumAccount(User):
    pass

In [ ]:
user_1 = User()
user_2 = GoldAccount()
user_3 = PlatinumAccount()

In [ ]:
# user_1.greet()
user_2.greet()
user_3.greet()

However, what if you need to "force" that all child classes implement some variant of the `greet` method?

In this situation you cannot.

<br>

A possible solution is to create a blueprint for the classes `GoldAccount` and `PlatinumAccount`.

This blueprint specifies which methods they must implement.

This avoids the complications caused by careless **multiple inheritance**:

In [ ]:
from abc import ABC, abstractmethod

In [ ]:
class User(ABC):
    
    @abstractmethod
    def greet(self):
        pass


class GoldAccount(User):
    
    def greet(self):
        print("Welcome to your Gold account")
        
        
class PlatinumAccount(User):
    
    def greet(self):
        print("Welcome to your Platinum account")

In [ ]:
user_1 = User()
user_2 = GoldAccount()
user_3 = PlatinumAccount()

Unlike with **inheritance**, with abstraction you cannot **instantiate abstract classes**.

You can only use them as a blueprint:

In [ ]:
user_2 = GoldAccount()
user_3 = PlatinumAccount()

In [ ]:
user_2.greet()
user_3.greet()

Another advantage over plain inheritance is that you cannot skip overriding methods:

In [ ]:
class User(ABC):
    
    @abstractmethod
    def greet(self):
        pass


class GoldAccount(User):
    
    def greet(self):
        print("Welcome to your Gold account")
        
        
class PlatinumAccount(User):
    pass

In [ ]:
user_2 = GoldAccount()
user_3 = PlatinumAccount()

The *interpreter* will immediately tell you that the abstract class has a `greet` method that you must also specify in your class.

<br>

### Abstract classes as blueprint and contract

---

In [ ]:
class User(ABC):
    def __init__(self, name: str, email: str):
        self.email = email
        self.name = name

    @abstractmethod
    def account_type(self):
        """Process the correct account type setting."""
        pass

Now you need to create different **account types**.

<br>

E.g. 3 different subscription tiers:
1. **Gold**, (100 USD/mo)
2. **Platinum**, (200 USD/mo),
3. **Diamond**, (300 USD/mo).

In [ ]:
class GoldAccount(User):
    GOLD_ACCOUNT_PRICE: int = 100
        
    def account_type(self):
        print(f"User: {self.name} chose Gold account, price: {self.GOLD_ACCOUNT_PRICE}.")

In [ ]:
class PlatinumAccount(User):
    PLATINUM_ACCOUNT_PRICE: int = 200
        
    def account_type(self):
        print(f"User: {self.name} chose Platinum account, price: {self.PLATINUM_ACCOUNT_PRICE}.")

In [ ]:
class DiamondAccount(User):
    DIAMOND_ACCOUNT_PRICE: int = 300

    def account_type(self):
        print(f"User: {self.name} chose Diamond account, price: {self.DIAMOND_ACCOUNT_PRICE}.")

In [ ]:
user_2 = GoldAccount("Luke", "luke.gulas@email.com")
user_2.account_type()

In [ ]:
user_3 = PlatinumAccount("John", "john.adam@email.com")
user_3.account_type()

In [ ]:
user_4 = DiamondAccount("Mark", "mark.honza@email.com")
user_4.account_type()

In [ ]:
user_5 = User("David", "david@smith.com")

<br>

`ABC` is a class you must inherit from the `abc` module. Python does not have **abstract classes built in like some other languages**.

<br>

You must also mark the method as abstract using the `@abstractmethod` decorator.

<br>

In the abstract method you don't write any implementation, only documentation and `pass`.

You also cannot create instances of an abstract class. It only serves as a blueprint.

In [ ]:
class Car(ABC):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    @abstractmethod
    def start_engine(self):
        """docstring"""
        pass
    
    @abstractmethod
    def turn_on_stereo(self):
        """docstring"""
        pass

In [ ]:
class PassengerCar(Car):
    pass

class Truck(Car):
    pass

<br>

### 🧠 EXERCISE 🧠 Create an abstract class `Parser`:

1. Define an abstract class `Parser` with two abstract methods: `check_extension`, `get_content`,
2. Create classes `TXTParser`, `CSVParser` and `JSONParser`,
3. all three classes implement in their own way the methods: `check_extension`, `get_content`,
4. the method `check_extension` returns `True` if it is the desired extension (typical for each parser) or `False`,
5. the method `get_content` first verifies that the file exists and has the correct extension, then returns the content.

In [ ]:
txt_parser = TXTParser()
notes = txt_parser.get_content('my_notes.txt')

<details>
    <summary>▶️ Solution</summary>
    
```python
import abc
import csv
import json
import pathlib

from pandas import read_csv


class Parser(abc.ABC):
    
    @abc.abstractmethod
    def check_extension(self, file: str):
        pass
    
    @abc.abstractmethod
    def get_content(self, source):
        pass


class TXTParser(Parser):

    def check_extension(self, file: str) -> bool:
        return pathlib.Path(file).suffix == '.txt'
        
    def get_content(self, file: str):
        if file and self.check_extension(file):
            with open(file) as txt:
                return txt.readlines()
        else:
            raise Exception("Cannot load the specified TXT file.")
    
    
class CSVParser(Parser):

    def check_extension(self, file: str) -> bool:
        return pathlib.Path(file).suffix == '.csv'
        
    def get_content(self, file: str):
        if file and self.check_extension(file):
            return read_csv(file, sep=';')
        else:
            raise Exception("Cannot load the specified CSV file.")


class JSONParser(Parser):
    
    def check_extension(self, file: str) -> bool:
        return pathlib.Path(file).suffix == '.json'
        
    def get_content(self, file: str):
        if file and self.check_extension(file):
            return json.load(file)
        else:
            raise Exception("Cannot load the specified JSON file.")
```
</details>

<br>

### 🧠 EXERCISE 🧠 Create a class `GameCharacter` and add the following:

- Define an abstract class `GameCharacter` that inherits from `ABC` in the `abc` library,
- the class `GameCharacter` has an instance attribute `_health` with value `100`,
- the class `GameCharacter` has two abstract methods `attack` and `defend`,
- define a class `Knight` that inherits from `GameCharacter` and has its own attribute `_health` with value `150`,
- define the `attack` method for `Knight` with the message: `Knight attacks with a sword!`,
- define the `defend` method for `Knight` with the message: `Knight defends with a shield!`,
- define a class `Mage` that inherits from `GameCharacter` and has its own attribute `_health` with value `80`,
- define the `attack` method for `Mage` with the message: `Mage attacks with magic!`,
- define the `defend` method for `Mage` with the message: `Mage defends with a magic shield!`.

In [ ]:
knight_matthew = Knight()
mage_luke = Mage()

<details>
    <summary>▶️ Solution</summary>
    
```python
from abc import ABC, abstractmethod


class GameCharacter(ABC):
    def __init__(self):
        self._health = 100

    @abstractmethod
    def attack(self):
        pass

    @abstractmethod
    def defend(self):
        pass


class Knight(GameCharacter):
    def __init__(self):
        super().__init__()
        self._health = 150

    def attack(self):
        print("Knight attacks with a sword!")

    def defend(self):
        print("Knight defends with a shield!")


class Mage(GameCharacter):
    def __init__(self):
        super().__init__()
        self._zivoty = 80

    def attack(self):
        print("Mage attacks with magic!")

    def defend(self):
        print("Mage defends with a magic shield!")
```
</details>

## Different types of classes

---

Not every class is the same as the previous one.

Very often you will encounter classes that focus on:
1. **data**, *attributes*,
2. **behaviour**, *methods* of classes.

There are also various combinations, i.e. the classes you have seen so far.

<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Ftse1.mm.bing.net%2Fth%3Fid%3DOIP.JQOcRw8TY0x1Rq5YzqzECAHaHa%26pid%3DApi&f=1&ipt=c632d7f3d5fad7ee0b75e0b50a42b7310978676ead0a911de8fbf1fb96f3b854&ipo=images" width="200" style="margin-left:auto; margin-right:auto">

## Data-oriented classes

---

Look at a class that **has no methods** besides attributes:

In [ ]:
class Book:
    def __init__(self, author: str, title: str, year: int) -> None:
        self.author = author
        self.title = title
        self.year = year

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993)

In [ ]:
print(heir_to_empire.author)

In [ ]:
print(heir_to_empire.__dict__)

If your program is based on manipulating and storing data, use **data classes**.

Whether you *serialize* data or work with data from a database, files or other sources, *data classes* let you work with this data in a structured and efficient way.

<br>

Rewriting the `Book` class as a *data class*:

In [ ]:
from dataclasses import dataclass

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993)

In [ ]:
print(heir_to_empire.__dict__)

In [ ]:
think_like_bee = Book("Roman Linhart", "Think Like a Bee", 2021)

In [ ]:
print(
    heir_to_empire.author,
    think_like_bee.year,
    sep='\n'
)

*Data classes* even generate the magic method `__repr__` for you:

In [ ]:
print(heir_to_empire, think_like_bee, sep='\n')

The only **obvious drawback** can be the slightly confusing treatment of attributes.

*Data classes* turn class attributes into *instance attributes*.

### Default values

---

For *data classes* you can add *default values*:

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int
    paperback: bool = True

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993)

In [ ]:
print(heir_to_empire)

For simple data types like `bool`, `int`, `float` and `str` nothing changes.

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int
    paperback: bool = True
    tags: list = ['sci-fi', 'star wars']

Using a **mutable default value** in a data class is not allowed.

A built-in exception will prevent the interpreter from running a script that would give all *instances* the same mutable value.

If you want to use a mutable type like a list, you can use the `field` object:

In [ ]:
from dataclasses import dataclass, field

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int
    paperback: bool = True
    tags: list = field(default_factory=list)

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993)

In [ ]:
heir_to_empire.tags.append('sci-fi')

In [ ]:
print(heir_to_empire)

You can also combine default arguments with custom objects.

For example, if you want to generate a unique `id` for each book:

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int
    paperback: bool = True
    tagy: list = field(default_factory=list)
    book_id: str = 'ACBDE123'

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993)
think_like_bee = Book("Roman Linhart", "Think Like a Bee", 2021)

In [ ]:
print(heir_to_empire, think_like_bee, sep='\n')

But that would not be practical; it is better to use a custom function:

In [ ]:
import uuid


def get_id() -> uuid.UUID:
    return uuid.uuid1()

In [ ]:
my_id = get_id()

In [ ]:
print(my_id)

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int
    paperback: bool = True
    tagy: list = field(default_factory=list)
    book_id: str = field(default_factory=get_id)

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993)
think_like_bee = Book("Roman Linhart", "Think Like a Bee", 2021)

In [ ]:
print(heir_to_empire, think_like_bee, sep='\n')

At this point the parameter `book_id` is still overridable:

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993, book_id="123456")

In [ ]:
print(heir_to_empire)

That is not entirely practical, because incorrect use of the data class `Book` can get you into trouble.

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int
    paperback: bool = True
    tagy: list = field(default_factory=list)
    book_id: str = field(init=False, default_factory=get_id)

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993, book_id="123456")

The *interpreter* will stop you here, because when instantiating it will not allow you to override the default value.

### Validation via magic method

---

If you need to add **validation**, you can add it to the data class:

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int
    paperback: bool = True
    tagy: list = field(default_factory=list)
    id_knihy: str = field(init=False, default_factory=vrat_id)
    
    def validate(self):
        if not isinstance(self.year, int):
            raise Exception("Parameter 'year' must be an integer")

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993)

In [ ]:
heir_to_empire.validate()

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", "1993")

In [ ]:
heir_to_empire.validate()

If you want smoother validation, use the `__post_init__` method:

In [ ]:
@dataclass
class Book:
    author: str
    title: str
    year: int
    paperback: bool = True
    tagy: list = field(default_factory=list)
    id_knihy: str = field(init=False, default_factory=vrat_id)
    
    def __post_init__(self):
        if not isinstance(self.year, int):
            raise Exception("Parameter 'year' must be an integer")

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", "1993")

In [ ]:
heir_to_empire = Book("Timothy Zahn", "Heir to the Empire", 1993)

<br>

### 🧠 EXERCISE 🧠 Create a class `User`:

1. Define a data class `User`,
2. the class User has attributes: `age`, `email`, `first_name`, `last_name`, `created` and `account_age`,
3. set the attribute `account_age` as a *default parameter* with value `None`,
4. create a magic method `__post_init__` that checks whether the age parameter is an integer greater than 0,
5. in `__post_init__` also compute how many days old the account is as of the current date and time,
6. create a status method that formats the string `<FIRST_NAME> <LAST_NAME>, account age: <AGE>`,
7. work with the prepared users below and print all users' statuses using a loop.

In [ ]:
user_1 = User(first_name='Matthew', age=21, last_name='Smith', created='22/2/2021', email='matthew@smith.com')
user_2 = User(first_name='Peter', age=20, last_name='Jones', created='19/5/2021', email='p.jones@gmail.com')
user_3 = User(first_name='Lucy', age=20, last_name='Brown', created='8/3/2020', email='lucy.brown@email.com')

<details>
    <summary>▶️ Solution</summary>
    
```python
import pandas
from dataclasses import dataclass, field

@dataclass
class User:
    age: int
    email: str
    first_name: str
    last_name: str
    created: str
    account_age: int = field(default=None)

    def __post_init__(self):
        if self.age is not None and self.age < 0:
            raise ValueError("Age must be a positive number")
                            
        rozdil = pandas.Timestamp.now() - pandas.Timestamp(
            pandas.to_datetime(self.vytvoreno, format='%d/%m/%Y')
        )
        self.stari_uctu = rozdil.days

    def status(self):
        return f"{self.first_name} {self.last_name}, account age: {self.account_age}"
```
</details>

<br>

[Questionnaire after lesson 4](https://forms.gle/Db2vba9W7riTCPF29)

---